# 03 — Il confronto (tentativo 11)

Per testata, fold per gruppo. Tre varianti, ognuna con le etichette NewsGuard **2026** e, per controllo, **2022**:

- **base**: solo il testo dell'articolo, tutte le 296 testate valutate;
- **titolo**: titolo (pulito dalla cornice del sito) + articolo, 296 testate;
- **politici**: training e test solo sulle testate con il solo argomento politico (143 valutate).

Modelli: **feature** (TF-IDF + LR, voto per testata) e **ModernBERT-base** (media delle probabilità per testata; la
media dei semi finiti). Ogni modello si valuta sulle etichette del suo anno; i modelli base 2022 anche sulle etichette
2026. I modelli base si valutano anche sulle 143 testate politiche, per confrontarli con la variante politici sulle
stesse testate.

Metriche: F1 macro sui 5 livelli, accuratezza, ±1 livello, binario (punteggio ≥ 60). Intervalli al 95% con un
**bootstrap per gruppo** (2000 ricampionamenti dei gruppi dell'insieme valutato); differenze sugli stessi
ricampionamenti, significative se l'intervallo non contiene lo zero. Si rifà dopo ogni modello finito.

In [ ]:
import json

import numpy as np
import pandas as pd

from comune import CLASSI, LIVELLI, RESULTS, VERSIONI, metriche_di, per_testata

B = 2000
OUT = RESULTS / "confronto"; OUT.mkdir(parents=True, exist_ok=True)
t = pd.read_csv(VERSIONI / "fold.csv").set_index("domain")
val = t[t.valutazione].sort_index()
pol = pd.read_csv(VERSIONI / "solo_politici.csv").set_index("domain")
INSIEMI = {"tutte": val.index, "politici": val.index[val.index.isin(pol.index[pol.valutazione])]}
VERO = {"2026": val.label, "2022": val.label_2022}
SUFFISSO = {"base": "", "titolo": "_titolo", "politici": "_politici"}
PRED = {}                                  # (variante, modello, anno) -> livello previsto per testata (Series su val.index)
for var, suf in SUFFISSO.items():
    for anno in ["2026", "2022"]:
        f = RESULTS / f"feature_tuned{suf}_{anno}_fold_gruppo_articoli.csv"
        if f.exists() and (RESULTS / f"feature_tuned{suf}_{anno}_FATTO").exists():
            PRED[(var, "feature", anno)] = per_testata(pd.read_csv(f)).y_pred.reindex(val.index)
        probs = []
        for seme in [42, 1, 2]:
            d = RESULTS / "transformer" / f"modernbert{suf}_{anno}_s{seme}"
            if (d / "FATTO").exists():
                probs.append(pd.read_csv(d / "articoli.csv").groupby("domain")[CLASSI].mean().reindex(val.index))
        if probs:
            p = sum(probs) / len(probs)
            PRED[(var, f"ModernBERT ({len(probs)} semi)" if len(probs) > 1 else "ModernBERT", anno)] = pd.Series(
                np.where(p.isna().any(axis=1), np.nan, p.fillna(0).to_numpy().argmax(1)), index=val.index)
assert PRED, "nessun modello finito"
print("modelli finiti:", list(PRED))

In [ ]:
CAMP = {}
for nome, idx in INSIEMI.items():
    g = val.gruppo[idx].to_numpy(); gruppi = np.unique(g); pos = {x: np.where(g == x)[0] for x in gruppi}
    rng = np.random.default_rng(0)
    CAMP[nome] = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]
METR = ["f1_macro", "accuratezza", "tolerant_1", "bin_accuratezza"]
BS = {}


def quattro(y, p):
    """Le 4 metriche del bootstrap, veloci (uguali a metriche_di: F1 macro sui 5 livelli con zero_division=0)."""
    M = np.bincount(y * 5 + p, minlength=25).reshape(5, 5)
    tp = np.diag(M); den = M.sum(0) + M.sum(1)
    f1 = np.divide(2 * tp, den, out=np.zeros(5), where=den > 0).mean()
    d = np.abs(y - p)
    return [f1, (d == 0).mean(), (d <= 1).mean(), ((y >= 2) == (p >= 2)).mean()]


def valuta(chiave, vero, insieme):
    idx = INSIEMI[insieme]
    y, p = VERO[vero][idx].to_numpy().astype(int), PRED[chiave][idx]
    assert not p.isna().any(), (chiave, insieme)
    p = p.to_numpy().astype(int)
    if (chiave, vero, insieme) not in BS:
        BS[(chiave, vero, insieme)] = np.array([quattro(y[i], p[i]) for i in CAMP[insieme]])
    m = metriche_di(y, p)
    assert np.allclose([m[k] for k in METR], quattro(y, p))          # controllo: stesse metriche
    return m, BS[(chiave, vero, insieme)]


def casi(chiave):
    var, mod, anno = chiave
    insiemi = ["politici"] if var == "politici" else ["tutte", "politici"] if var == "base" else ["tutte"]
    veri = [anno, "2026"] if (var == "base" and anno == "2022") else [anno]
    return [(v, i) for i in insiemi for v in veri]


righe = []
for chiave in PRED:
    for vero, insieme in casi(chiave):
        m, b = valuta(chiave, vero, insieme)
        r = {"variante": chiave[0], "modello": chiave[1], "addestrato su": chiave[2], "valutato su": vero, "testate": insieme, "n": len(INSIEMI[insieme])}
        for j, k in enumerate(METR):
            r[k] = m[k]; r[f"{k} 2.5%"] = np.percentile(b[:, j], 2.5); r[f"{k} 97.5%"] = np.percentile(b[:, j], 97.5)
        righe.append(r)
tab = pd.DataFrame(righe).set_index(["variante", "modello", "addestrato su", "valutato su", "testate"]); tab.to_csv(OUT / "intervalli.csv")
tab[["n"] + METR].round(3)

## Le differenze

In [ ]:
def trova(var, famiglia, anno):
    k = [c for c in PRED if c[0] == var and c[1].startswith(famiglia) and c[2] == anno]
    return k[0] if k else None


coppie = []
for fam in ["feature", "ModernBERT"]:
    a, b = trova("base", fam, "2026"), trova("base", fam, "2022")
    if a and b:
        coppie += [(a, "2026", b, "2022", "tutte", f"{fam}: etichette 2026 − 2022 (ognuno sul suo anno)"),
                   (a, "2026", b, "2026", "tutte", f"{fam}: addestrato su 2026 − su 2022 (valutati sul 2026)")]
    for anno in ["2026", "2022"]:
        a, b = trova("titolo", fam, anno), trova("base", fam, anno)
        if a and b:
            coppie.append((a, anno, b, anno, "tutte", f"{fam}, {anno}: titolo + articolo − solo articolo"))
        a, b = trova("politici", fam, anno), trova("base", fam, anno)
        if a and b:
            coppie.append((a, anno, b, anno, "politici", f"{fam}, {anno}: addestrato solo sui politici − su tutte (143 testate politiche)"))
for var in SUFFISSO:
    for anno in ["2026", "2022"]:
        a, b = trova(var, "ModernBERT", anno), trova(var, "feature", anno)
        if a and b:
            ins = "politici" if var == "politici" else "tutte"
            coppie.append((a, anno, b, anno, ins, f"ModernBERT − feature ({var}, {anno})"))
righe = []
for a, va, b, vb, ins, nome in coppie:
    (ma, ba), (mb, bb) = valuta(a, va, ins), valuta(b, vb, ins)
    d = ba - bb; r = {"confronto": nome, "testate": ins}
    for j, k in enumerate(METR):
        r[f"Δ {k}"] = ma[k] - mb[k]; r[f"Δ {k} 2.5%"] = np.percentile(d[:, j], 2.5); r[f"Δ {k} 97.5%"] = np.percentile(d[:, j], 97.5)
    righe.append(r)
diff = pd.DataFrame(righe).set_index("confronto") if righe else pd.DataFrame()
diff.to_csv(OUT / "differenze.csv")
diff[[c for c in diff.columns if "f1_macro" in c or "bin_" in c]].round(3) if len(diff) else "nessuna coppia ancora"

## Matrici di confusione (righe: vero; colonne: predetto), ogni modello sul suo anno e sul suo insieme

In [ ]:
for chiave, p in PRED.items():
    var, mod, anno = chiave
    idx = INSIEMI["politici" if var == "politici" else "tutte"]
    c = pd.crosstab(VERO[anno][idx].map(dict(enumerate(LIVELLI))).rename("vero"), p[idx].astype(int).map(dict(enumerate(LIVELLI))).rename("predetto"))
    c = c.reindex(index=LIVELLI, columns=LIVELLI, fill_value=0)
    c.to_csv(OUT / f"confusione_{var}_{mod.split(' (')[0]}_{anno}.csv")
(OUT / "aggiornato.txt").write_text(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))
print("matrici salvate in", OUT)